# DSL para Criação de Histórias em Quadrinhos

Uma linguagem de domínio específico implementada em **Guile Scheme** que permite descrever histórias em quadrinhos de forma declarativa.

## Como funciona:

A DSL usa funções Scheme para compor elementos visuais e narrativos que geram HTML/CSS como output:

```
Cena (nome, local) 
  → Quadro (número, layout)
    → Cenário (imagem de fundo)
    → Personagens (posição, emoção)
    → Falas (personagem, ação, texto)
    → Efeitos (nome do efeito)
    → Recordatórios (legendas/captions)
```

Cada elemento é uma **função Scheme** que gera uma string HTML.

# ## Funções Disponíveis

| Função | Parâmetros | Descrição |
|--------|-----------|-----------|
| `(cena nome local . bodies)` | nome, local, conteúdo | Define uma cena |
| `(quadro num layout . bodies)` | número, layout, conteúdo | Define um quadro |
| `(cenario nome)` | nome | Define o fundo |
| `(personagem nome pos emo)` | nome, posição, emoção | Adiciona personagem |
| `(fala nome acao texto)` | nome, ação, diálogo | Adiciona fala |
| `(efeito nome)` | nome | Adiciona efeito |
| `(recordatorio texto)` | texto | Adiciona legenda |

**Posições:** "esquerda", "direita", "centro"  
**Ações:** "diz", "grita", "pensa", "sussurra"

In [ ]:
;; ==============================================
;; INTERPRETADOR SCHEME PARA DSL DE QUADRINHOS
;; ==============================================

(define (ensure-string x)
  (cond
    ((string? x) x)
    ((symbol? x) (symbol->string x))
    (else (error "Esperado string ou símbolo" x))))

(define (char-safe c)
  (if (or (char-alphabetic? c) (char-numeric? c))
      (string (char-downcase c))
      "_"))

(define (normalize s)
  (apply string-append
         (map char-safe (string->list (ensure-string s)))))

(define (bg-path name)
  (string-append "assets/backgrounds/" (normalize name) ".png"))

(define (char-path name emo)
  (string-append "assets/characters/"
                 (normalize name) "/"
                 (normalize emo) ".png"))

(define (effect-path name)
  (string-append "assets/effects/" (normalize name) ".png"))

(define (map-position pos)
  (let ((p (ensure-string pos)))
    (cond
     ((string=? p "esquerda") "left:20px; bottom:20px;")
     ((string=? p "direita")  "right:20px; bottom:20px;")
     ((string=? p "centro")   "left:50%; transform:translateX(-50%); bottom:20px;")
     (else "left:20px; bottom:20px;"))))

(define (cena nome local . bodies)
  (string-append
   "<div class='scene'><h2>"
   (ensure-string nome) " - " (ensure-string local)
   "</h2>"
   (apply string-append bodies)
   "</div>"))

(define (quadro num layout . bodies)
  (string-append
   "<div class='panel'>"
   (apply string-append bodies)
   "</div>"))

(define (cenario nome)
  (string-append
   "<div class='background' style='background-image:url(\""
   (bg-path nome)
   "\")'></div>"))

(define (personagem nome pos emo)
  (string-append
   "<div class='character' style='" (map-position pos) "'>"
   "<img src='" (char-path nome emo) "'/>"
   "</div>"))

(define (fala personagem-nome acao texto)
  (let ((acao-str (ensure-string acao))
        (char-str (ensure-string personagem-nome)))
    (string-append
     "<div class='speech' data-action='" acao-str "' data-character='" char-str "'><b>"
     char-str
     " (" acao-str ")</b>: "
     (ensure-string texto)
     "</div>")))

(define (efeito nome)
  (string-append
   "<div class='effect'>"
   "<img src='" (effect-path nome) "'/>"
   "</div>"))

(define (recordatorio texto)
  (string-append
   "<div class='caption'>"
   (ensure-string texto)
   "</div>"))

(define css "
  body { font-family: Arial; background: #111; color: #fff; padding: 20px; }
  .scene { margin-bottom: 40px; }
  .panel {
    width: 500px; height: 500px;
    position: relative; overflow: hidden;
    border: 2px solid #fff; margin: 10px;
    background: #333;
  }
  .background {
    position: absolute; inset: 0;
    background-size: cover; background-position: center;
    z-index: 0;
  }
  .character { position: absolute; z-index: 5; }
  .character img { width: 200px; }
  .effect { position: absolute; z-index: 4; bottom: 20px; right: 20px; }
  .effect img { width: 100px; }
  .speech {
    position: absolute; z-index: 6;
    background: white; color: black;
    padding: 10px; border-radius: 20px;
    max-width: 200px; font-size: 12px;
    border: 2px solid black;
  }

  .speech[data-character='Joao'] {
    top: 80px; left: 20px;
  }

  .speech[data-character='Maria'] {
    top: 80px; right: 10px;
  }
  .speech::after {
    content: '';
    position: absolute;
    bottom: -10px;
    left: 20px;
    border-width: 10px;
    border-style: solid;
    border-color: white transparent transparent transparent;
  }
  /* Balão de pensamento em formato de nuvem */
  .speech[data-action='pensa'] {
    border-radius: 40px 40px 40px 40px;
    border: 2px solid black;
    background: white;
    color: black;
    padding: 15px 20px;
    position: relative;
  }

  /* Círculo maior (bolinha principal) */
  .speech[data-action='pensa']::before {
    content: '';
    position: absolute;
    width: 20px;
    height: 20px;
    background: white;
    border: 2px solid black;
    border-radius: 50%;
    bottom: -15px;
    left: 15px;
  }

  /* Círculo menor (bolinha menor) */
  .speech[data-action='pensa']::after {
    content: '';
    position: absolute;
    width: 12px;
    height: 12px;
    background: white;
    border: 2px solid black;
    border-radius: 50%;
    bottom: -28px;
    left: 8px;
  }
  .caption {
    background: #000;
    padding: 5px;
    font-style: italic;
    position: absolute;
    z-index: 10;
    top: 0;
    left: 0;
    right: 0;
    width: 100%;
    box-sizing: border-box;
  }
")

# ## Exemplo 1: "O Despertar do Poder"

Uma cena simples da floresta com um personagem:

In [ ]:
(define exemplo1
  (cena "O Despertar do Poder" "Floresta Proibida"
    (quadro 1 "Topo_Largo"
      (cenario "floresta")
      (personagem "Joao" "esquerda" "em_guarda")
      (fala "Joao" "pensa" "Eu sinto uma presença... ela está perto.")
      (efeito "CRACK")
      (recordatorio "O silêncio da floresta é interrompido."))))

(display exemplo1)

# ## Exemplo 2: "Reencontro"

Uma cena com múltiplos personagens em diálogo:

In [ ]:
(define exemplo2
  (cena "Reencontro" "Cafeteria da Cidade"
    (quadro 1 "Normal"
      (cenario "cafeteria")
      (personagem "Joao" "esquerda" "em_guarda")
      (personagem "Maria" "direita" "feliz")
      (fala "Joao" "diz" "Quanto tempo, Maria!")
      (fala "Maria" "diz" "Que coincidência te encontrar por aqui!")
      (recordatorio "Um reencontro inesperado na cafeteria."))))

(display exemplo2)

In [ ]:
;; ========================================
;; EXEMPLO 2 COM MACRO
;; ========================================

(define exemplo2-macro
  (cena-dialogo-completa "Reencontro" "Cafeteria da Cidade" "cafeteria"
    ("Joao" "esquerda" "em_guarda" "Quanto tempo, Maria!")
    ("Maria" "direita" "feliz" "Que coincidência te encontrar por aqui!")
    "Um reencontro inesperado na cafeteria."))

(display exemplo2-macro)

In [ ]:
;; ========================================
;; EXEMPLO 1 COM MACRO
;; ========================================

(define exemplo1-macro
  (cena-dramatica "O Despertar do Poder" "Floresta Proibida"
                  "floresta"
                  "Joao" "esquerda" "em_guarda"
                  "pensa"
                  "Eu sinto uma presença... ela está perto."
                  "CRACK"
                  "O silêncio da floresta é interrompido."))

(display exemplo1-macro)

# O Uso de Macros

Macros são uma forma de **reduzir repetição de código** transformando código em tempo de compilação. Em Scheme, usamos `define-syntax` e `syntax-rules` para criar padrões reutilizáveis.

## Por que usar macros?

- Cenas que exigem 7 linhas ficam com 1
- Código mais legível
- Menos erros: Estrutura garantida pela macro

## Exemplo Prático: Redução de Código

SEM MACRO (7 linhas):
```scheme
(cena "O Despertar" "Floresta"
  (quadro 1 "Normal"
    (cenario "floresta")
    (personagem "João" "esquerda" "medo")
    (fala "João" "diz" "Algo está acontecendo...")))
```

COM MACRO (1 linha):
```scheme
(cena-simples "O Despertar" "Floresta" "floresta" "João" "esquerda" "medo" "Algo está acontecendo...")
```

In [ ]:
;; ========================================
;; DEFININDO AS MACROS
;; ========================================

;; MACRO 1: CENA SIMPLES
;; Reduz uma cena com 1 quadro, 1 personagem e 1 fala para 1 linha
(define-syntax cena-simples
  (syntax-rules ()
    ((cena-simples nome local cenario personagem posicao emocao fala)
     (cena nome local
       (quadro 1 "Normal"
         (cenario cenario)
         (personagem personagem posicao emocao)
         (fala personagem "diz" fala))))))

;; MACRO 2: CENA COM DOIS PERSONAGENS EM DIÁLOGO
;; Combina dois personagens + suas falas em 1 chamada
(define-syntax cena-dialogo
  (syntax-rules ()
    ((cena-dialogo nome local cenario (p1 pos1 em1 fala1) (p2 pos2 em2 fala2))
     (cena nome local
       (quadro 1 "Normal"
         (cenario cenario)
         (personagem p1 pos1 em1)
         (personagem p2 pos2 em2)
         (fala p1 "diz" fala1)
         (fala p2 "diz" fala2))))))

;; MACRO 3: CENA DRAMÁTICA (com efeito e narração)
;; Para cenas com 1 personagem, ação dramática, efeito e narração
(define-syntax cena-dramatica
  (syntax-rules ()
    ((cena-dramatica nome local cenario personagem posicao emocao acao fala nome-efeito naracao)
     (cena nome local
       (quadro 1 "Normal"
         (cenario cenario)
         (personagem personagem posicao emocao)
         (fala personagem acao fala)
         (efeito nome-efeito)
         (recordatorio naracao))))))

;; MACRO 4: DIÁLOGO COMPLETO (2 personagens + narração)
;; Estende cena-dialogo adicionando recordatorio
(define-syntax cena-dialogo-completa
  (syntax-rules ()
    ((cena-dialogo-completa nome local cenario (p1 pos1 em1 fala1) (p2 pos2 em2 fala2) naracao)
     (cena nome local
       (quadro 1 "Normal"
         (cenario cenario)
         (personagem p1 pos1 em1)
         (personagem p2 pos2 em2)
         (fala p1 "diz" fala1)
         (fala p2 "diz" fala2)
         (recordatorio naracao))))))

In [ ]:
;; ========================================
;; EXEMPLO 2 COM MACRO
;; ========================================

(define exemplo2-macro
  (cena-dialogo-completa "Reencontro" "Cafeteria da Cidade" "cafeteria"
    ("Joao" "esquerda" "em_guarda" "Quanto tempo, Maria!")
    ("Maria" "direita" "feliz" "Que coincidência te encontrar por aqui!")
    "Um reencontro inesperado na cafeteria."))

(display exemplo2-macro)

In [ ]:
;; ========================================
;; EXEMPLO 1 COM MACRO
;; ========================================

(define exemplo1-macro
  (cena-dramatica "O Despertar do Poder" "Floresta Proibida"
                  "floresta"
                  "Joao" "esquerda" "em_guarda"
                  "pensa"
                  "Eu sinto uma presença... ela está perto."
                  "CRACK"
                  "O silêncio da floresta é interrompido."))

(display exemplo1-macro)